In [7]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.data_download import download_data
import json 
import pandas as pd
import pickle
import os

data_dir = Path("../data")

In [8]:
# List the categories to download here
categories = ["All_Beauty"]
reload_data = False

if reload_data:
    for category in categories:
        download_data(category)


In [9]:
for category in categories:
    review_file = f"../data/raw/{category}.jsonl"
    metadata_file = f"../data/raw/meta_{category}.jsonl"

    print(f"Data exploration for {category} dataset:")

    # Read reviews
    reviews = []
    with open(review_file, "r") as f:
        for line in f:
            reviews.append(json.loads(line))

    # Read metadata
    meta = []
    with open(metadata_file, "r") as f:
        for line in f:
            meta.append(json.loads(line))

    print("Reviews count:", len(reviews))
    print("Metadata count:", len(meta))

    print("Review fields:", list(reviews[0].keys()))
    print("Metadata fields:", list(meta[0].keys()))

    print("First review:", reviews[0])
    print("First metadata record:", meta[0])


Data exploration for All_Beauty dataset:
Reviews count: 701528
Metadata count: 112590
Review fields: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
Metadata fields: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
First review: {'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': Tr

In [10]:
from langchain_core.documents import Document
from src.bm25 import text_tokenizer, build_bm25, bm25_search

tokenized_corpus_file = os.path.join(data_dir, "processed", "tokenized_corpus.pkl")

documents = []
tokenized_corpus = []

meta_title_lookup = {
    item["parent_asin"]: item["title"]
    for item in meta
}

if os.path.exists(tokenized_corpus_file):
    print("Loading tokenized corpus...")
    
    with open(tokenized_corpus_file, "rb") as f:
        tokenized_corpus = pickle.load(f)

else:
    print("Generating tokenized corpus...")
    
    for review in reviews:
        tokens = text_tokenizer(review["text"])
        tokenized_corpus.append(tokens)

    with open(tokenized_corpus_file, "wb") as f:
        pickle.dump(tokenized_corpus, f)

# Build documents (needed for LangChain BM25)
for tokens, review in zip(tokenized_corpus, reviews):
    processed_text = " ".join(tokens)
    product_title = meta_title_lookup.get(review["parent_asin"], "")
    combined_text = product_title + " " + processed_text
    documents.append(
        Document(
            page_content=combined_text,
            metadata={"asin": review["parent_asin"],
            "product_title": product_title
            }
        )
    )


Loading tokenized corpus...


In [11]:
bm25_index_path = "../data/processed/bm25_index.pkl"


if os.path.exists(bm25_index_path):

    print("Loading existing BM25 index...")

    with open(bm25_index_path, "rb") as f:
        bm25 = pickle.load(f)

else:

    print("Building BM25 index...")

    bm25 = build_bm25(documents)

    with open(bm25_index_path, "wb") as f:
        pickle.dump(bm25, f)

Building BM25 index...


In [ ]:
# This is how you pass a query to the BM25 search

query = "wireless bluetooth headphones"

results = bm25_search(bm25, documents, query, k=5)

for doc, score in results:
    print("Score:", score)
    print("Product:", doc.metadata.get("product_title"))
    print("ASIN:", doc.metadata.get("asin"))
    print()

Score: 35.066935752274375
Product: Wireless Earbuds, 455D Stereo Sound Wireless Headphones Wireless Sport Earbud with Breathing Mini in-Ear Sports Earphones Noise Cancelling Headsets, Bluetooth Earbuds
ASIN: B014VTGC9I

Score: 31.791503280805657
Product: Yontune Sleep Headphones Headband Timing Wireless Cozy Band Washable for Running Workout Unique Gifts (Lengthened), Black grey
ASIN: B0B533WQVC

Score: 21.152462808027963
Product: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Dimmable Light Detachable 10X Magnification Rechargable Power (Rose Gold)
ASIN: B0769VLLW6

Score: 20.74221611796488
Product: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Dimmable Light Detachable 10X Magnification Rechargable Power (Rose Gold)
ASIN: B0769VLLW6

Score: 20.5431920076002
Product: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Di

In [15]:
texts = [doc.page_content for doc in documents]
texts

['Herbivore - Natural Sea Mist Texturizing Salt Spray (Coconut, 8 oz) spray really nice smells really good goes really fine does trick i will say feels like you need lot though get texture i want i have lot hair medium thickness i am comparing other brands yucky chemicals so im gonna stick try',
 'All Natural Vegan Dry Shampoo Powder - Eco Friendly, Root Touch Up | Hair Powder Volumizer | For Brown Hair, Brunette and Dark Hair. (Brun + Application Brush) Two Goats Apothecary product does what i need do i just wish was odorless or had soft coconut smell having my head smell like orange coffee offputting granted i did know smell was described but i was hoping would light',
 'New Road Beauty - Creamsicle - Variety 3 Pack Paraffin Wax - Peach, Vanilla & Mango - Moisturize and Smooths Skin - Nourishing and Provides Massage Therapy smells good feels great',
 'muaowig Ombre Body Wave Bundles 1B Grey Human Hair Bundles 8 10 12 inch Hair Weave Body Wave Brazilian Remy Hair Bundles Ombre Hair Ex

In [ ]:
# This is how you pass a query to the semantic search

from src.semantic import semantic_search

query = "wireless bluetooth headphones"

results = semantic_search(documents, query, k=5, sample_size=10000, reload_index=True)

for doc, score in results:
    print("Score:", score)
    print("Product:", doc.metadata.get("product_title"))
    print("ASIN:", doc.metadata.get("asin"))

Score: 1.0159401
Product: Earbuds Ear Buds Sport Earbuds Running Earbuds in Ear Headphones Wired Earphones with Microphone Mic Stereo and Volume Control Waterproof Wired Earphone Android Mp3 Players Tablet Laptop 3.5mm Audio
ASIN: B07KQ4XNDV
Score: 1.1151774
Product: MMUSS Sleep Headphones Headband with Ultra Thin Stereo Speakers.Perfect for Sleeping,Sports,Air Travel,Meditation and Relaxation(Black2)
ASIN: B07JQ136H9
Score: 1.1721325
Product: Gaming Headphones Chamvict Xbox One Headset with Stereo Sound Ps4 Headset with Mic Noise Canceling for PS4,PC,Laptop,Cell Phone,Xbox,Find Enemies Before They Find You
ASIN: B07RTK8GR1
Score: 1.1949987
Product: COSYOO Fashion Boho Headbands for Women 12PCS Hair Band Elastic Vintage Button Knotted Headbands Head Wrap Floral Bandeau Headbands
ASIN: B09BN5TYK5
Score: 1.1954854
Product: Nxconsu 12Pcs Headbands for Women Girls Teens Elastic Hair Bands Silky Solid Colors Pleated Crinkle Head Band Fashion Trendy Outfit Colorful Thin Comfortable Hair Acce

In [19]:
# This is how you pass a query to the BM25 search

query = "wireless bluetooth headphones"

results = bm25_search(bm25, documents, query, k=5)

for doc, score in results:
    print("Score:", score)
    print("Product:", doc.metadata.get("product_title"))
    print("ASIN:", doc.metadata.get("asin"))
    print()

Score: 35.066935752274375
Product: Wireless Earbuds, 455D Stereo Sound Wireless Headphones Wireless Sport Earbud with Breathing Mini in-Ear Sports Earphones Noise Cancelling Headsets, Bluetooth Earbuds
ASIN: B014VTGC9I

Score: 31.791503280805657
Product: Yontune Sleep Headphones Headband Timing Wireless Cozy Band Washable for Running Workout Unique Gifts (Lengthened), Black grey
ASIN: B0B533WQVC

Score: 21.152462808027963
Product: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Dimmable Light Detachable 10X Magnification Rechargable Power (Rose Gold)
ASIN: B0769VLLW6

Score: 20.74221611796488
Product: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Dimmable Light Detachable 10X Magnification Rechargable Power (Rose Gold)
ASIN: B0769VLLW6

Score: 20.5431920076002
Product: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Di